# Hello Agent: Step-by-Step Tutorial

In this notebook you will build a minimal tool-using agent in 5 steps:

1. Load environment and imports
2. Verify API key
3. Define a tool
4. Create a model client and agent
5. Run a prompt and inspect the answer

## Step 1: Load dependencies and environment variables

In [11]:
import os
from pathlib import Path
from dotenv import load_dotenv
from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root
# Student's note: the .env file should be located in the parent directory of the current working directory to ensure that the OPENAI_API_KEY is properly loaded for use in the OpenAIChatCompletionClient.

load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

## Step 2: Verify Your API Key

This check helps you catch environment issues before running model calls.

In [12]:
# Step 2 (code): confirm OPENAI_API_KEY is available
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is empty. Check your .env file path and contents.")

print("API key loaded successfully.")

API key loaded successfully.


## Step 3: Define A Simple Tool

The tool is a normal Python function. The agent can call it when the prompt needs weather information.

In [13]:
# Step 3 (code): define a simple weather tool
def get_weather(location: str) -> str:
    """Return a mock weather report for the provided location."""
    return f"The weather in {location} is sunny, 15C"

## Step 4: Create The Client And Agent

Here you connect to the model and register your tool with the agent.

In [14]:
# Step 4 (code): create model client and tool-using agent
client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)

agent = Agent(
    name="assistant",
    instructions="You are helpful. Use tools when appropriate.",
    model_client=client,
    tools=[get_weather]
)

## Step 5: Run A Prompt

This call should trigger the agent to use the weather tool for Berlin.

In [15]:
# Step 5 (code): run the agent
prompt = "What's the weather in Berlin?"
response = await agent.run(prompt)
print(response.messages[-1].content)

The weather in Berlin is sunny and 15°C.


In [16]:
# Use the agent to get weather information
async for event in agent.run_stream("What's the weather like in Berlin?"):
    print(event)

[user] 09:27:46 | What's the weather like in Berlin?
[assistant] 09:27:49 | tool_call: get_weather(location=Berlin, Germany)
[assistant] 09:27:49 | tool_response: ✓ The weather in Berlin, Germany is sunny, 15C
[assistant] 09:27:49 | The weather in Berlin, Germany is sunny, 15C
[assistant] 09:27:57 | Right now in Berlin it's sunny and about 15°C (59°F). A light jacket should be fine. Would you like an hourly forecast, a 7-day outlook, or precipitation chances?
[user] 09:27:46 | What's the weather like in Berlin?
[assistant] 09:27:49 | [calling tools: get_weather(location=Berlin, Germany)]
[assistant] 09:27:49 | The weather in Berlin, Germany is sunny, 15C
[assistant] 09:27:57 | Right now in Berlin it's sunny and about 15°C (59°F). A light jacket should be fine. Would you like an hourly forecast, a 7-day outlook, or precipitation chances?

[usage] duration: 11.4s, tokens: in:532, out:457 | finish: stop
